In [2]:
!pip install pythainlp

In [3]:
!pip install transformers

In [4]:
!pip install llama-index-llms-huggingface
!pip install llama-index

In [5]:
pip install bitsandbytes accelerate

In [1]:
!pip install -U bitsandbytes

In [6]:
%pip install llama-index-embeddings-huggingface

In [7]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from pythainlp.tokenize import word_tokenize

from transformers import AutoTokenizer
import torch
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core import PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [8]:
!pip install -q gradio

In [9]:
import llama_index

In [10]:
pip install huggingface_hub==0.36.2

In [11]:
from huggingface_hub import  notebook_login
notebook_login()

In [12]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


#System Prompt

In [13]:
from llama_index.core import PromptTemplate

template = """You are a knowledgeable and precise assistant specialized in answering questions based on academic and research-based sources, specifically regarding โรคติดต่อทางเพศสัมพันธ์ (STDs).

Instructions:
1. **Comprehension and Accuracy**: Carefully read and comprehend the provided context to ensure accuracy in your response.
2. **Truthfulness**: If the context lacks sufficient information to answer, respond with "ไม่มีข้อมูลในระบบครับตอนนี้".
3. **Contextual Relevance**: Use only information from the provided context. Avoid adding any outside information or assumptions.

Important:
- If no context is provided, respond with "ไม่มีข้อมูลในระบบครับตอนนี้".
- Ensure that answers are relevant, accurate, and concise.

\nQuestion: {question} \nContext: {context} \nAnswer (in Thai):"""
query_wrapper_prompt = PromptTemplate("<|USER|>{query_str}<|ASSISTANT|>")

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
from llama_index.core import SimpleDirectoryReader

required_exts = [".md"]

reader = SimpleDirectoryReader(
    input_dir="/content/drive/MyDrive/mdffile",
    required_exts=required_exts,
    recursive=True,
)
documents = reader.load_data()

In [16]:
documents[0]

Document(id_='0b8f6cd0-6ac7-4b81-8251-ef7a964a4d7d', embedding=None, metadata={'file_path': '/content/drive/MyDrive/mdffile/testalldoc1.md', 'file_name': 'testalldoc1.md', 'file_type': 'text/markdown', 'file_size': 61485, 'creation_date': '2026-06-09', 'last_modified_date': '2024-11-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# โรคติดต่อทางเพศสัมพันธ์ (STDs)\r\n\r\n## บทนำ\r\nโรคติดต่อทางเพศสัมพันธ์ (STDs) หรือการติดเชื้อจากเพศสัมพันธ์ (STIs) เป็นการติดเชื้อที่แพร่กระจายผ่านการมีเพศสัมพันธ์เป็นหลัก โรคเหล่านี้เกิดจากเชื้อไวรัส แบคทีเรีย และปรสิต ที่สามารถแพร่จากคนสู่คนผ่านของเหลวในร่างกาย เช่น เลือด น้ำอสุจิ ของเหลวจากช่องคลอด แล

#LLM model

In [17]:
from transformers import AutoTokenizer

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
stopping_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

In [18]:
from llama_index.llms.huggingface import HuggingFaceLLM
import torch

llm = HuggingFaceLLM(
    context_window=8192,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.6, "do_sample": False},
    system_prompt=template,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name=model_id,
    model_name=model_id,
    device_map="auto",
    stopping_ids=stopping_ids,
    tokenizer_kwargs={"max_length": 4096},
    model_kwargs={
        "dtype": torch.float16,
        "load_in_4bit": True
    }
)

Settings.llm = llm

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
text_splitter = SentenceSplitter(chunk_size=1024 ,chunk_overlap = 20 )
Settings.text_splitter = text_splitter
Settings.llm = llm
Settings.embed_model = embed_model

In [20]:
index = VectorStoreIndex.from_documents(documents ,transformations=[text_splitter])

In [21]:
prompt_template = PromptTemplate(template=template ,
                                 template_var_mappings = {"query_str" : "question" , "context_str" : "context"})

In [22]:
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5  # Top-k documents to retrieve
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer()

# Assemble query engine with retriever and response synthesizer
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

# Update query engine with custom prompt
query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": prompt_template}
)

def predict(input):
    print(f"Querying LLM with input: {input}")
    response = query_engine.query(input)
    print(f"Received response: {response}")
    return str(response)


In [23]:
thai_query = "สาเหตุของหนองในแท้"
output = predict(thai_query)
print("output: ",output)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: สาเหตุของหนองในแท้


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Received response: สาเหตุของหนองในแท้คือการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) |<|END_ASSISTANT|>ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบ
output:  สาเหตุของหนองในแท้คือการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) |<|END_ASSISTANT|>ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่มีข้อมูลในระบบครับตอนนี้|<|END_ASSISTANT|> ไม่ม

In [24]:
thai_query = "ฉันมีอาการรอยแตกและแดงบริเวณรอบอวัยวะเพศ คือโรคติดต่อทางเพศสัมพันธ์ใด"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: ฉันมีอาการรอยแตกและแดงบริเวณรอบอวัยวะเพศ คือโรคติดต่อทางเพศสัมพันธ์ใด
Received response: ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับ
output:  ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระ

In [25]:
thai_query = "อยากรู้เรื่องการป้องกันทั่วไป"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: อยากรู้เรื่องการป้องกันทั่วไป
Received response: การป้องกันการติดเชื้อโรคติดต่อทางเพศสัมพันธ์สามารถทำได้ด้วยการหลีกเลี่ยงการมีเพศสัมพันธ์โดยไม่สวมถุงยางอนามัย การตรวจเลือดเพื่อตรวจหาการติดเชื้อเป็นประจำ และการใช้ถุงยางอนามัยทุกครั้งที่มีเพศสัมพันธ์ นอกจากนี้ยังสามารถป้องกันได้โดยการฉีดวัคซีนเพื่อป้องกันการติดเชื้อเอชไอวี (HIV) และการตรวจสุขภาพเป็นประจำเพื่อตรวจหาการติดเชื้อโรคติดต่อทางเพศสัมพันธ์อย่างสม่ำเสมอ

การป้องกันการติดเชื้อโรคติดต่อทางเพศสัมพันธ์สามารถทำได้ด้วยการหลีกเลี่ยงการมีเพศสัมพันธ์โดยไม่สวมถุงยางอนามัย การตรวจเลือดเพื่อตรวจหาการติดเชื้อเป็นประจำ และการใช้ถุงยางอนามัยทุกครั้งที่ม
output:  การป้องกันการติดเชื้อโรคติดต่อทางเพศสัมพันธ์สามารถทำได้ด้วยการหลีกเลี่ยงการมีเพศสัมพันธ์โดยไม่สวมถุงยางอนามัย การตรวจเลือดเพื่อตรวจหาการติดเชื้อเป็นประจำ และการใช้ถุงยางอนามัยทุกครั้งที่มีเพศสัมพันธ์ นอกจากนี้ยังสามารถป้องกันได้โดยการฉีดวัคซีนเพื่อป้องกันการติดเชื้อเอชไอวี (HIV) และการตรวจสุขภาพเป็นประจำเพื่อตรวจหาการติดเชื้อโรคติดต่อทางเพศสัมพันธ์อย่างสม่ำเสมอ


In [26]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?
Received response: โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?

โรคหนองในแท้ (Gonorrhea) และ โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เป็นโรคติดต่อทางเพศสัมพันธ์ที่มีอาการคล้ายคลึงกัน แต่สาเหตุและอาการที่แสดงออกแตกต่างกัน

โรคหนองในแท้ (Gonorrhea) เกิดจากการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) ซึ่งเป็นเชื้อโรคที่แพร่กระจายผ่านการมีเพศสัมพันธ์ทางอวัยวะเพศ, ทวารหนัก หรือปาก

โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เกิดจากการติดเชื้อแบคทีเรียชนิดอื่นที่ไม่ใช่ *Neisseria gonorrhoeae* เช่น *Chlamydia trachomatis*, *Ureaplasma parvum*, *Mycoplasma genitalium*, *Trichomonas vaginalis*
output:  โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?

โรคหนองในแท้ (Gonorrhea) และ โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เป็นโรคติดต่อทางเพศสัมพันธ์ที่มีอาการคล้ายคลึงกัน แต่สาเหตุและอาการที่แสดงออกแตกต่างกัน

โรคหนองในแท้ (Gonorrhea) เกิดจากการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) ซึ่งเป็นเชื้อโรคที่แพร่

In [27]:
thai_query = "โรคติดต่อทางเพศสัมพันธ์ ที่รักษาไม่หาย"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: โรคติดต่อทางเพศสัมพันธ์ ที่รักษาไม่หาย
Received response: ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับ
output:  ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบ

In [28]:
thai_query = "การติดเชื้อ Chlamydia สามารถหายได้เองโดยไม่ต้องรักษาหรือไม่"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: การติดเชื้อ Chlamydia สามารถหายได้เองโดยไม่ต้องรักษาหรือไม่
Received response: ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับ
output:  ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนน

In [29]:
thai_query = "โรคติดต่อทางเพศสัมพันธ์ (STDs) แพร่กระจายอย่างไร และวิธีการแพร่เชื้อหลักคืออะไร"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: โรคติดต่อทางเพศสัมพันธ์ (STDs) แพร่กระจายอย่างไร และวิธีการแพร่เชื้อหลักคืออะไร
Received response: ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับ
output:  ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มีข้อมูลในระบบครับตอนนี้ | ไม่มี

In [30]:
thai_query = "อาการของหนองในเทียมขเพศชาย"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: อาการของหนองในเทียมขเพศชาย
Received response: อาการของหนองในเทียมขเพศชาย

อาการของหนองในเทียมขเพศชายอาจรวมถึง:

- มีมูกหรือน้ำใสไหลออกจากอวัยวะเพศ
- ปวดหรือแสบเมื่อปัสสาวะ (Dysuria)
- ปวดที่ถุงอัณฑะ
- อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่ขา, หรือปวดที่หน้าท้อง

อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่ขา, หรือปวดที่หน้าท้อง

อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่ขา, หรือปวดที่หน้าท้อง

อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่
output:  อาการของหนองในเทียมขเพศชาย

อาการของหนองในเทียมขเพศชายอาจรวมถึง:

- มีมูกหรือน้ำใสไหลออกจากอวัยวะเพศ
- ปวดหรือแสบเมื่อปัสสาวะ (Dysuria)
- ปวดที่ถุงอัณฑะ
- อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่ขา, หรือปวดที่หน้าท้อง

อาการเหล่านี้อาจไม่แสดงออกทันที แต่อาจมีอาการอื่น ๆ เช่น ปวดท้องน้อย, ปวดที่ขา, หรือปวดที่หน้าท้อง

อาการเหล่านี้อาจไม่แสดงออกทันที แ

In [31]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Querying LLM with input: โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?
Received response: โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?

โรคหนองในแท้ (Gonorrhea) และ โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เป็นโรคติดต่อทางเพศสัมพันธ์ที่มีอาการคล้ายคลึงกัน แต่สาเหตุและอาการที่แสดงออกแตกต่างกัน

โรคหนองในแท้ (Gonorrhea) เกิดจากการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) ซึ่งเป็นเชื้อโรคที่แพร่กระจายผ่านการมีเพศสัมพันธ์ทางอวัยวะเพศ, ทวารหนัก หรือปาก

โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เกิดจากการติดเชื้อแบคทีเรียชนิดอื่นที่ไม่ใช่ *Neisseria gonorrhoeae* เช่น *Chlamydia trachomatis*, *Ureaplasma parvum*, *Mycoplasma genitalium*, *Trichomonas vaginalis*
output:  โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?

โรคหนองในแท้ (Gonorrhea) และ โรคหนองในเทียม (Non-gonococcal urethritis - NGU) เป็นโรคติดต่อทางเพศสัมพันธ์ที่มีอาการคล้ายคลึงกัน แต่สาเหตุและอาการที่แสดงออกแตกต่างกัน

โรคหนองในแท้ (Gonorrhea) เกิดจากการติดเชื้อแบคทีเรีย (Neisseria gonorrhoeae) ซึ่งเป็นเชื้อโรคที่แพร่

In [37]:
import gradio as gr

gr.ChatInterface(predict).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 1 arguments for function <function predict at 0x7f525a73d3a0>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1060: UserWarning: Expected maximum 1 arguments for function <function predict at 0x7f525a73d3a0>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 1 arguments for function <function predict at 0x7f525a73d440>, received 2.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1060: UserWarning: Expected maximum 1 arguments for function <function predict at 0x7f525a73d440>, receive

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a3cb9fbfb5412ccfc0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
